# LANCE-seq Figure 5

从空间转录组 AnnData 输入开始，透明复现 Figure 5 的18张分析图。

Transparent, top-to-bottom reproduction of the 18 Figure 5 analysis panels from a spatial-transcriptomics AnnData input.

### 中文
导入依赖，并集中定义输入、富集表目录、输出目录、样本映射、固定基因集与随机种子。

### English
Import dependencies and define input, enrichment-table and output paths, sample mapping, fixed gene sets, and random seed.

In [ ]:
# Imports and configuration
import os
import re
import sys
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

CODE_DIR = Path(os.environ.get("FIGURE5_CODE_DIR", Path.cwd())).resolve()
INPUT_H5AD = Path(os.environ.get("FIGURE5_INPUT_H5AD", "../data/LANCE8.h5ad"))
ENRICHMENT_DIR = Path(os.environ.get("FIGURE5_ENRICHMENT_DIR", "data"))
OUTPUT_DIR = Path(os.environ.get("FIGURE5_OUTPUT_DIR", str(CODE_DIR / "outputs")))
RANDOM_SEED = 0
VOLCANO_JITTER_SEED = 42  # preserves the established display-only ceiling jitter

SAMPLE_MAP = {"1": "MN1", "2": "MN2", "3": "MN3", "4": "CON1", "5": "MN4", "6": "CON2", "7": "MN5", "8": "CON3"}
PLOT_ORDER = ["MN1", "MN5", "CON3", "MN2", "MN4", "CON2"]
CV_GENES = ["Rgn", "Slc1a2", "Cyp2c29", "Gulo"]
PV_GENES = ["Cyp2f2", "Sds", "Hal", "Pck1"]
HELDOUT_CV = ["Cyp2e1", "Glul", "Cyp7a1"]
HELDOUT_PV = ["Ass1", "Alb"]
APAP_SPATIAL_GENES = ["Hmox1", "Ddit3", "Atf3", "Fgf21"]
ZONATION_MARKER_GENES = ["Glul", "Cyp2e1", "Cyp2f2", "Sds"]
APAP_MODULES = {
    "Oxidative stress / NRF2": ["Hmox1", "Nqo1", "Gclc", "Gclm", "Srxn1", "Txnrd1", "Slc7a11", "Gsta1", "Gsta2", "Gstp1"],
    "General / ER stress": ["Atf3", "Ddit3", "Atf4", "Hspa1a", "Hspa1b", "Gadd45a", "Gadd45b"],
    "Inflammatory / chemokine": ["Tnf", "Il1b", "Nfkbia", "Ccl2", "Cxcl1", "Cxcl2", "Ptgs2", "Socs3"],
    "Cell injury / death": ["Ddit3", "Atf4", "Gadd45a", "Gadd45b", "Bbc3", "Bax", "Casp3", "Fas", "Trp53inp1"],
}
RELATIVE_STATES = ["Relative PV-like", "Intermediate", "Relative CV-like"]
STATE_COLORS = {"Relative CV-like": "#D98C8C", "Intermediate": "#D7D7D7", "Relative PV-like": "#7EA6D8"}
SAMPLE_COLORS = {"MN1": "#AFC7E8", "MN5": "#83A9D6", "CON3": "#8F8F8F", "MN2": "#E6B0AA", "MN4": "#D58F86", "CON2": "#B56B62"}
PAIRS = {"MN1_to_MN2": ("MN1", "MN2"), "MN5_to_MN4": ("MN5", "MN4"), "CON3_to_CON2": ("CON3", "CON2")}

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import figure5_utils as f5

np.random.seed(RANDOM_SEED)
f5.configure_style(RANDOM_SEED)
OUTPUT_DIR = f5.prepare_output(OUTPUT_DIR)
print("Input:", INPUT_H5AD)
print("Enrichment tables:", ENRICHMENT_DIR)
print("Output:", OUTPUT_DIR)


### 中文
直接读取 AnnData，按 batch 提取六个样本的 count-like X、空间坐标，并对所需基因执行每 spot 10,000 计数归一化与 log1p。

### English
Read AnnData directly, extract count-like X and spatial coordinates for six samples, and normalize selected genes to 10,000 counts per spot followed by log1p.

In [2]:
# Load AnnData and extract sample-level count matrices, coordinates, and normalized genes
required_genes = list(dict.fromkeys(
    CV_GENES + PV_GENES + HELDOUT_CV + HELDOUT_PV + APAP_SPATIAL_GENES +
    ZONATION_MARKER_GENES + [g for genes in APAP_MODULES.values() for g in genes] +
    ["Cxcl2", "Gdf15", "Nqo1", "Cyp1a2"]
))

adata = sc.read_h5ad(INPUT_H5AD, backed="r")
if "batch" not in adata.obs or "spatial" not in adata.obsm:
    raise ValueError("The h5ad must contain obs['batch'] and obsm['spatial'].")

batch_labels = adata.obs["batch"].astype(str).str.strip()
var_names = adata.var_names.astype(str).to_numpy()
frames, counts, coords, obs_names = {}, {}, {}, {}
sample_rows = []
for batch_id, sample in SAMPLE_MAP.items():
    if sample not in PLOT_ORDER:
        continue
    mask = (batch_labels == batch_id).to_numpy()
    sub = adata[mask, :].to_memory()
    sub.var_names_make_unique()
    X = sub.X.tocsr() if sparse.issparse(sub.X) else sparse.csr_matrix(np.asarray(sub.X))
    integer_like = float(np.isclose(X.data, np.rint(X.data), atol=1e-8, rtol=0).mean())
    if X.nnz and (X.data.min() < 0 or integer_like < 0.98):
        raise ValueError(f"{sample} X is not nonnegative count-like data.")
    available = [gene for gene in required_genes if gene in sub.var_names]
    missing_core = [gene for gene in CV_GENES + PV_GENES if gene not in available]
    if missing_core:
        raise ValueError(f"Missing core zonation genes in {sample}: {missing_core}")
    indices = sub.var_names.get_indexer(available)
    normalized = f5.normalize_selected(X, indices)  # counts/spot -> 10,000 -> log1p
    frames[sample] = pd.DataFrame(normalized, index=sub.obs_names.astype(str), columns=available)
    counts[sample] = X
    coords[sample] = np.asarray(sub.obsm["spatial"], dtype=float)
    obs_names[sample] = pd.Index(sub.obs_names.astype(str))
    sample_rows.append({"sample": sample, "batch": batch_id, "n_spots": sub.n_obs, "n_genes": sub.n_vars, "integer_like_fraction": integer_like})
adata.file.close()
sample_summary = pd.DataFrame(sample_rows).set_index("sample").loc[PLOT_ORDER].reset_index()
print(sample_summary.to_string(index=False))


sample batch  n_spots  n_genes  integer_like_fraction
   MN1     1     1226    24082               0.982741
   MN5     7      948    24082               1.000000
  CON3     8     1929    24082               1.000000
   MN2     2     1845    24082               1.000000
   MN4     5      932    24082               1.000000
  CON2     6     1879    24082               1.000000


### 中文
合并 MN1/MN2 原始计数，直接运行 Scanpy Wilcoxon，计算校正 P 值、log2FC、阈值分类并绘制火山图。

### English
Combine MN1/MN2 raw counts, run Scanpy Wilcoxon directly, calculate adjusted P values, log2FC and thresholds, and plot the volcano.

In [3]:
# MN2 versus MN1 differential expression and Fig. 5b volcano
X_mn12 = sparse.vstack([counts["MN1"], counts["MN2"]], format="csr")
obs_mn12 = pd.DataFrame(index=pd.Index(np.concatenate([obs_names["MN1"], obs_names["MN2"]]), name="spot_id"))
obs_mn12["sample"] = np.concatenate([np.repeat("MN1", len(obs_names["MN1"])), np.repeat("MN2", len(obs_names["MN2"]))])
obs_mn12["deg_group"] = obs_mn12["sample"].map({"MN1": "MN1_Healthy0h", "MN2": "MN2_APAP6h"})
adata_deg = ad.AnnData(X=X_mn12, obs=obs_mn12, var=pd.DataFrame(index=pd.Index(var_names, name="gene")))
sc.pp.filter_genes(adata_deg, min_cells=3)
sc.pp.normalize_total(adata_deg, target_sum=1e4)
sc.pp.log1p(adata_deg)
adata_deg.obs["deg_group"] = pd.Categorical(adata_deg.obs["deg_group"], categories=["MN1_Healthy0h", "MN2_APAP6h"])
sc.tl.rank_genes_groups(adata_deg, groupby="deg_group", groups=["MN2_APAP6h"], reference="MN1_Healthy0h", method="wilcoxon", use_raw=False, pts=True, tie_correct=True)
deg = sc.get.rank_genes_groups_df(adata_deg, group="MN2_APAP6h").rename(columns={"names": "gene"})
deg["minus_log10_padj_raw"] = -np.log10(np.maximum(deg["pvals_adj"].to_numpy(float), 1e-300))
noise_pattern = re.compile(r"^mt-|^Gm|-ps|Rik$|^Hba-|^Hbb-|^Rp[sl]\d+", re.IGNORECASE)
deg["volcano_included"] = ~deg["gene"].astype(str).map(lambda value: bool(noise_pattern.search(value)))
deg["significance"] = "Not Sig"
deg.loc[deg["volcano_included"] & (deg["pvals_adj"] < 0.05) & (deg["logfoldchanges"] > 1), "significance"] = "Up"
deg.loc[deg["volcano_included"] & (deg["pvals_adj"] < 0.05) & (deg["logfoldchanges"] < -1), "significance"] = "Down"
deg["logfoldchanges_plot"] = deg["logfoldchanges"].clip(-8, 8)
deg["minus_log10_padj_plot"] = deg["minus_log10_padj_raw"].clip(upper=200)
ceiling = deg["minus_log10_padj_raw"] > 200
deg.loc[ceiling, "minus_log10_padj_plot"] = 200 + np.random.default_rng(VOLCANO_JITTER_SEED).uniform(-1, 3, ceiling.sum())

plot_df = deg.loc[deg["volcano_included"]].copy()
priority = ["Hmox1", "Nqo1", "Atf3", "Ddit3", "Fgf21", "Gdf15", "Cyp2e1"]
label_rows = []
for direction in ["Up", "Down"]:
    subset = plot_df.loc[plot_df["significance"] == direction].sort_values(["pvals_adj", "scores"], ascending=[True, False])
    label_rows.append(pd.concat([subset.loc[subset["gene"].isin(priority)], subset]).drop_duplicates("gene").head(5))
labels = pd.concat(label_rows, ignore_index=True)

fig, ax = plt.subplots(figsize=(9, 8))
for status, color, size, alpha in [("Not Sig", "#E0E0E0", 13, 0.48), ("Down", "#1F77B4", 24, 0.78), ("Up", "#D62728", 24, 0.78)]:
    subset = plot_df.loc[plot_df["significance"] == status]
    ax.scatter(subset["logfoldchanges_plot"], subset["minus_log10_padj_plot"], s=size, c=color, alpha=alpha, edgecolors="none", label=status)
ax.axvline(1, color="black", linestyle="--", linewidth=0.8, alpha=0.65)
ax.axvline(-1, color="black", linestyle="--", linewidth=0.8, alpha=0.65)
ax.axhline(1.30103, color="black", linestyle="--", linewidth=0.8, alpha=0.65)
ax.set_xlim(-8.2, 8.2); ax.set_ylim(0, 220)
levels = [205, 216, 210, 203, 214]
for direction in ["Down", "Up"]:
    subset = labels.loc[labels["significance"] == direction].sort_values("logfoldchanges_plot")
    for rank, row in enumerate(subset.itertuples(index=False)):
        ax.scatter(row.logfoldchanges_plot, row.minus_log10_padj_plot, s=36, facecolor="white", edgecolor="black", linewidth=0.7, zorder=4)
        ax.annotate(row.gene, (row.logfoldchanges_plot, row.minus_log10_padj_plot), xytext=(float(np.clip(row.logfoldchanges_plot, -7.65, 7.65)), levels[rank % len(levels)]), textcoords="data", ha="center", fontsize=8.2, arrowprops=dict(arrowstyle="-", color="#555555", lw=0.45))
ax.set_xlabel("Log2 fold change\n(MN2 APAP 6 h / MN1 Healthy 0 h)")
ax.set_ylabel("−log10 adjusted P-value")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=3)
ax.grid(False); fig.tight_layout()
f5.save_png(OUTPUT_DIR, fig, "Fig5B_MN2_vs_MN1_Volcano.png"); plt.close(fig)
print({"tested": len(deg), "up": int((deg.significance == "Up").sum()), "down": int((deg.significance == "Down").sum())})


{'tested': 23797, 'up': 1239, 'down': 2954}


### 中文
读取本地 GO 表，显式计算 overlap ratio 与 −log10 adjusted P-value，选择固定代表条目并绘制上、下调图。

### English
Read local GO tables, explicitly calculate overlap ratio and −log10 adjusted P value, select fixed representative terms, and plot up/down panels.

In [4]:
# GO enrichment tables and Fig. 5c / Fig. S5a
go_files = {"up": "Fig5C_GO_Up_full.csv", "down": "Fig5C_GO_Down_full.csv"}
go_terms = {
    "up": ["Inflammatory Response", "Neutrophil Chemotaxis", "Cytokine-Mediated Signaling Pathway", "Chemokine-Mediated Signaling Pathway", "Regulation Of Angiogenesis", "Apoptotic Process", "Integrated Stress Response Signaling", "Intrinsic Apoptotic Signaling Pathway In Response To Endoplasmic Reticulum Stress"],
    "down": ["Cholesterol Biosynthetic Process", "DNA Metabolic Process", "Mitochondrial Translation", "DNA Repair", "Mitochondrial Gene Expression", "Lipid Biosynthetic Process", "Purine Nucleotide Metabolic Process", "acetyl-CoA Metabolic Process"],
}
go_tables = {}
for direction in ["up", "down"]:
    table = pd.read_csv(ENRICHMENT_DIR / go_files[direction])
    overlap = table["Overlap"].astype(str).str.extract(r"(?P<hit>\d+)\s*/\s*(?P<total>\d+)").astype(float)
    table["Overlap_Ratio"] = overlap["hit"] / overlap["total"].replace(0, np.nan)
    table["minus_log10_padj"] = -np.log10(np.maximum(table["Adjusted P-value"].astype(float), 1e-300))
    table["Term_without_ID"] = table["Term"].astype(str).str.replace(r"\s*\(GO:\d+\)$", "", regex=True)
    requested = {term.casefold() for term in go_terms[direction]}
    go_tables[direction] = table.loc[table["Term_without_ID"].str.casefold().isin(requested)].sort_values("Adjusted P-value")
f5.plot_enrichment(go_tables["up"], OUTPUT_DIR, "Fig5C1_GO_Up.png", "Representative non-redundant GO terms | Up", "#D98C8C", "GO term")
f5.plot_enrichment(go_tables["down"], OUTPUT_DIR, "Fig5C2_GO_Down.png", "Representative non-redundant GO terms | Down", "#7EA6D8", "GO term")


### 中文
读取本地 KEGG 表，执行相同的 overlap 与校正 P 值处理，并绘制固定上、下调通路。

### English
Read local KEGG tables, apply the same overlap and adjusted-P processing, and plot the fixed up/down pathways.

In [5]:
# KEGG enrichment tables and Fig. S5b/c
kegg_files = {"up": "FigS5A_KEGG_Up_full.csv", "down": "FigS5A_KEGG_Down_full.csv"}
kegg_terms = {
    "up": ["Cytokine-cytokine receptor interaction", "IL-17 signaling pathway", "MAPK signaling pathway", "Hematopoietic cell lineage", "NF-kappa B signaling pathway", "TNF signaling pathway", "Chemokine signaling pathway"],
    "down": ["Drug metabolism", "Steroid biosynthesis", "Retinol metabolism", "Selenocompound metabolism", "Glutathione metabolism", "Purine metabolism", "Metabolism of xenobiotics by cytochrome P450"],
}
kegg_tables = {}
for direction in ["up", "down"]:
    table = pd.read_csv(ENRICHMENT_DIR / kegg_files[direction])
    overlap = table["Overlap"].astype(str).str.extract(r"(?P<hit>\d+)\s*/\s*(?P<total>\d+)").astype(float)
    table["Overlap_Ratio"] = overlap["hit"] / overlap["total"].replace(0, np.nan)
    table["minus_log10_padj"] = -np.log10(np.maximum(table["Adjusted P-value"].astype(float), 1e-300))
    table["Term_without_ID"] = table["Term"].astype(str).str.replace(r"\s*\(GO:\d+\)$", "", regex=True)
    requested = {term.casefold() for term in kegg_terms[direction]}
    kegg_tables[direction] = table.loc[table["Term_without_ID"].str.casefold().isin(requested)].sort_values("Adjusted P-value")
f5.plot_enrichment(kegg_tables["up"], OUTPUT_DIR, "FigS5A1_KEGG_Up.png", "Representative non-redundant KEGG pathways | Up", "#D98C8C", "KEGG pathway")
f5.plot_enrichment(kegg_tables["down"], OUTPUT_DIR, "FigS5A2_KEGG_Down.png", "Representative non-redundant KEGG pathways | Down", "#7EA6D8", "KEGG pathway")


### 中文
直接提取四个 APAP 基因和四个肝分区 marker 的 MN1/MN2 表达，绘制空间图及描述性箱线图。

### English
Directly extract MN1/MN2 expression for four APAP genes and four liver-zonation markers, then plot spatial maps and descriptive boxplots.

In [6]:
# APAP injury and liver-zonation marker expression
apap_expression = {sample: frames[sample][APAP_SPATIAL_GENES].copy() for sample in ["MN1", "MN2"]}
zonation_marker_expression = {sample: frames[sample][ZONATION_MARKER_GENES].copy() for sample in ["MN1", "MN2"]}
f5.plot_spatial_gene_grid(frames, coords, APAP_SPATIAL_GENES, OUTPUT_DIR, "Fig5D_APAP_Injury_Gene_Spatial_Maps_fixed.png")
f5.plot_box_grid(frames, APAP_SPATIAL_GENES, OUTPUT_DIR, "FigS5B_APAP_Marker_Expression_Distribution_fixed.png")
f5.plot_spatial_gene_grid(frames, coords, ZONATION_MARKER_GENES, OUTPUT_DIR, "Fig5E_Liver_Zonation_Marker_Spatial_Maps.png")
f5.plot_box_grid(frames, ZONATION_MARKER_GENES, OUTPUT_DIR, "FigS5C_Zonation_Marker_Expression_Distribution.png")
print("APAP genes:", APAP_SPATIAL_GENES)
print("Zonation markers:", ZONATION_MARKER_GENES)


APAP genes: ['Hmox1', 'Ddit3', 'Atf3', 'Fgf21']
Zonation markers: ['Glul', 'Cyp2e1', 'Cyp2f2', 'Sds']


### 中文
在每个样本内部逐基因 z-score，计算 CV 均值减 PV 均值，并用稳定排序分成三个等大小相对分区。

### English
Within each sample, z-score each core gene, calculate mean CV minus mean PV score, and use stable ranks to form three equal-size relative strata.

In [7]:
# Relative zonation score and rank-based tertiles
relative = {}
relative_rows = []
for sample in PLOT_ORDER:
    expr = frames[sample]
    core_genes = CV_GENES + PV_GENES
    detection = (expr[core_genes] > 0).mean()
    gene_sd = expr[core_genes].std(ddof=0)
    usable = [gene for gene in core_genes if detection[gene] >= 0.05 and gene_sd[gene] > 1e-8]
    used_cv = [gene for gene in CV_GENES if gene in usable]
    used_pv = [gene for gene in PV_GENES if gene in usable]
    if len(used_cv) < 3 or len(used_pv) < 3:
        raise ValueError(f"Insufficient core genes in {sample}: CV={used_cv}; PV={used_pv}")
    gene_z = (expr[usable] - expr[usable].mean()) / expr[usable].std(ddof=0)
    table = pd.DataFrame(index=expr.index)
    table["CV_relative_score"] = gene_z[used_cv].mean(axis=1)
    table["PV_relative_score"] = gene_z[used_pv].mean(axis=1)
    table["Relative_zonation_score"] = table["CV_relative_score"] - table["PV_relative_score"]
    table["Relative_state"] = f5.rank_tertiles(table["Relative_zonation_score"])
    relative[sample] = table
    relative_rows.append(table.assign(spot_id=table.index, sample=sample).reset_index(drop=True))
relative_all = pd.concat(relative_rows, ignore_index=True)
print(relative_all.groupby(["sample", "Relative_state"]).size().unstack(fill_value=0))


Relative_state  Intermediate  Relative CV-like  Relative PV-like
sample                                                          
CON2                     626               627               626
CON3                     643               643               643
MN1                      409               409               408
MN2                      615               615               615
MN4                      311               311               310
MN5                      316               316               316


### 中文
绘制连续相对分区分数与三状态空间图；共同显示范围仅统一子图尺度，不改变空间坐标。

### English
Plot continuous relative-zonation scores and three-state maps; common display limits only harmonize subplot scale and do not alter coordinates.

In [8]:
# Continuous and three-state relative zonation maps
spatial_limits, nearest_neighbor_spacing = f5.common_spatial_limits(coords, PLOT_ORDER)
pooled_scores = np.concatenate([relative[s]["Relative_zonation_score"] for s in PLOT_ORDER])
color_limit = float(np.max(np.abs(np.quantile(pooled_scores, [0.01, 0.99]))))
fig, axes = plt.subplots(2, 3, figsize=(10.4, 6.7))
for ax, sample in zip(axes.ravel(), PLOT_ORDER):
    im = ax.scatter(coords[sample][:, 0], coords[sample][:, 1], c=relative[sample]["Relative_zonation_score"], s=10, cmap="RdBu_r", vmin=-color_limit, vmax=color_limit, linewidths=0)
    f5.clean_spatial_axis(ax, sample)
    ax.set_xlim(*spatial_limits[sample][0]); ax.set_ylim(*spatial_limits[sample][1])
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02, label="Relative zonation score\nPV-like direction ← 0 → CV-like direction")
fig.suptitle("Within-sample relative zonation-associated ordering")
fig.subplots_adjust(top=0.92, right=0.9, wspace=0.08, hspace=0.17)
f5.save_png(OUTPUT_DIR, fig, "FigS5D_Relative_Zonation_Continuous_Spatial_Maps_fixed.png"); plt.close(fig)

fig, axes = plt.subplots(2, 3, figsize=(10.4, 6.7))
for ax, sample in zip(axes.ravel(), PLOT_ORDER):
    colors = [STATE_COLORS[state] for state in relative[sample]["Relative_state"]]
    ax.scatter(coords[sample][:, 0], coords[sample][:, 1], c=colors, s=10, linewidths=0)
    f5.clean_spatial_axis(ax, sample)
    ax.set_xlim(*spatial_limits[sample][0]); ax.set_ylim(*spatial_limits[sample][1])
handles = [Line2D([0], [0], marker="o", ls="none", mfc=STATE_COLORS[state], mec="none", label=state.replace("Relative ", ""), ms=6) for state in RELATIVE_STATES]
fig.legend(handles=handles, frameon=False, loc="center right")
fig.suptitle("Relative within-sample zonation states\nZones are relative transcriptional states, not direct anatomical assignments")
fig.subplots_adjust(top=0.88, right=0.86, wspace=0.08, hspace=0.17)
f5.save_png(OUTPUT_DIR, fig, "Fig5F_Relative_Zonation_3State_Spatial_Maps_fixed.png"); plt.close(fig)


### 中文
用未参与分区分数构建的 held-out CV/PV 基因，在健康样本中计算三区中位表达和连续分数 Spearman 相关。

### English
Using held-out CV/PV genes excluded from score construction, calculate stratum medians and Spearman correlations with the continuous score in healthy samples.

In [9]:
# Held-out validation; these genes do not enter the relative zonation score
heldout_rows = []
for sample in PLOT_ORDER:
    for gene in HELDOUT_CV + HELDOUT_PV:
        rho = float(spearmanr(relative[sample]["Relative_zonation_score"], frames[sample][gene]).statistic)
        for state in RELATIVE_STATES:
            values = frames[sample].loc[relative[sample]["Relative_state"] == state, gene]
            heldout_rows.append({"sample": sample, "Gene": gene, "Relative_state": state, "median_expression": values.median(), "Spearman_rho": rho})
heldout_validation = pd.DataFrame(heldout_rows)
healthy_samples = ["MN1", "MN5", "CON3"]
heldout_genes = HELDOUT_CV + HELDOUT_PV
fig, axes = plt.subplots(2, 3, figsize=(11.2, 7.0))
for ax, gene in zip(axes.ravel()[:5], heldout_genes):
    data = heldout_validation.query("sample in @healthy_samples and Gene == @gene")
    sns.pointplot(data=data, x="Relative_state", y="median_expression", order=RELATIVE_STATES, hue="sample", hue_order=healthy_samples, palette=SAMPLE_COLORS, markers="o", linestyles="-", ax=ax)
    ax.set_title(gene); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=25)
    ax.set_ylabel("Median log-normalized expression"); ax.legend(frameon=False, fontsize=6)
rho_table = heldout_validation.query("sample in @healthy_samples").drop_duplicates(["sample", "Gene"]).pivot(index="Gene", columns="sample", values="Spearman_rho").reindex(index=heldout_genes, columns=healthy_samples)
sns.heatmap(rho_table, annot=True, fmt=".2f", center=0, cmap="vlag", ax=axes.ravel()[5])
axes.ravel()[5].set_title("Spearman rho with relative score")
fig.suptitle("Healthy held-out marker validation of relative zonation")
fig.tight_layout()
f5.save_png(OUTPUT_DIR, fig, "FigS5E_Healthy_HeldOut_Zonation_Validation.png"); plt.close(fig)
print(rho_table.round(3).to_string())


sample    MN1    MN5   CON3
Gene                       
Cyp2e1  0.907  0.821  0.861
Glul    0.810  0.716  0.701
Cyp7a1  0.694  0.347  0.335
Ass1   -0.788 -0.615 -0.591
Alb    -0.785 -0.536 -0.613


### 中文
以 MN1 与 MN5 等权建立健康参考，方差明确包含样本内与样本间两部分，并计算绝对 CV/PV 程序分数。

### English
Build an equally weighted MN1/MN5 healthy reference whose variance explicitly includes within- and between-sample components, then calculate absolute CV/PV program scores.

In [10]:
# Healthy-reference absolute zonation programs with corrected variance
absolute_genes = CV_GENES + PV_GENES
mu1 = frames["MN1"][absolute_genes].mean()
mu5 = frames["MN5"][absolute_genes].mean()
var1 = frames["MN1"][absolute_genes].var(ddof=1)
var5 = frames["MN5"][absolute_genes].var(ddof=1)
mu_ref = (mu1 + mu5) / 2
within_var = (var1 + var5) / 2
between_var = (mu1 - mu5) ** 2 / 4
sd_ref = np.sqrt((within_var + between_var).clip(lower=1e-16))

def absolute_scores(expr, center, scale):
    z = (expr[absolute_genes] - center[absolute_genes]) / scale[absolute_genes]
    out = pd.DataFrame(index=expr.index)
    out["CV_reference_score"] = z[CV_GENES].mean(axis=1)
    out["PV_reference_score"] = z[PV_GENES].mean(axis=1)
    out["Reference_balance"] = out["CV_reference_score"] - out["PV_reference_score"]
    return out

absolute = {sample: absolute_scores(frames[sample], mu_ref, sd_ref) for sample in ["MN1", "MN5", "MN2", "MN4"]}
con_mu = frames["CON3"][absolute_genes].mean()
con_sd = frames["CON3"][absolute_genes].std(ddof=1).clip(lower=1e-8)
absolute.update({sample: absolute_scores(frames[sample], con_mu, con_sd) for sample in ["CON3", "CON2"]})
absolute_all = pd.concat([absolute[s].assign(sample=s) for s in PLOT_ORDER], ignore_index=True)

display_data = absolute_all.query("sample in ['MN1','MN5','MN2','MN4']")
display_order = ["MN1", "MN5", "MN2", "MN4"]
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.0))
for ax, column, title in zip(axes, ["CV_reference_score", "PV_reference_score", "Reference_balance"], ["Absolute CV-associated program", "Absolute PV-associated program", "Reference balance"]):
    sns.violinplot(data=display_data, x="sample", y=column, order=display_order, hue="sample", palette=SAMPLE_COLORS, legend=False, inner="quartile", cut=0, linewidth=0.7, ax=ax)
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=25)
fig.suptitle("Healthy-reference absolute program remodeling | spot-level descriptive distributions")
fig.tight_layout()
f5.save_png(OUTPUT_DIR, fig, "FigS5G_HealthyReference_Absolute_Zonation_Program_Remodeling.png"); plt.close(fig)

cv_values = np.concatenate([absolute[s]["CV_reference_score"] for s in display_order])
vmin, vmax = np.quantile(cv_values, [0.01, 0.99])
fig, axes = plt.subplots(1, 4, figsize=(11.7, 3.3))
for ax, sample in zip(axes, display_order):
    im = ax.scatter(coords[sample][:, 0], coords[sample][:, 1], c=absolute[sample]["CV_reference_score"], s=10, cmap="Reds", vmin=vmin, vmax=vmax, linewidths=0)
    f5.clean_spatial_axis(ax, sample)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02, label="Absolute CV-associated score\nrelative to corrected Healthy MN reference")
fig.suptitle("Absolute CV-associated spatial program maps")
fig.subplots_adjust(top=0.82, right=0.9, wspace=0.08)
f5.save_png(OUTPUT_DIR, fig, "FigS5H_Absolute_CV_Associated_Program_Spatial_Maps.png"); plt.close(fig)
print(pd.DataFrame({"mu_ref": mu_ref, "within_var": within_var, "between_var": between_var, "sd_ref": sd_ref}).round(4))


         mu_ref  within_var  between_var  sd_ref
Rgn      2.2759      0.0975       0.2003  0.5457
Slc1a2   1.0323      0.2446       0.0044  0.4990
Cyp2c29  2.1643      0.0850       0.1483  0.4830
Gulo     1.5034      0.1453       0.0000  0.3812
Cyp2f2   1.8826      0.1808       0.0009  0.4263
Sds      1.2620      0.1706       0.0875  0.5080
Hal      1.8787      0.2046       0.0941  0.5465
Pck1     2.5234      0.0748       0.9425  1.0086


### 中文
在每个 sample×相对分区内汇总原始计数，换算 CPM 和 log2(CPM+1)，计算各纵向/支持性样本对的 APAP−baseline 效应量。

### English
Within each sample×relative stratum, aggregate raw counts, convert to CPM and log2(CPM+1), and calculate APAP−baseline effects for primary and supportive sample pairs.

In [11]:
# Zone-stratified count aggregation and transcriptional effect sizes
def zone_effect_table(pair_name, state):
    baseline, apap = PAIRS[pair_name]
    mask_baseline = relative[baseline]["Relative_state"].to_numpy() == state
    mask_apap = relative[apap]["Relative_state"].to_numpy() == state
    count_baseline = np.asarray(counts[baseline][mask_baseline, :].sum(axis=0)).ravel()
    count_apap = np.asarray(counts[apap][mask_apap, :].sum(axis=0)).ravel()
    cpm_baseline = count_baseline / count_baseline.sum() * 1e6
    cpm_apap = count_apap / count_apap.sum() * 1e6
    keep = (np.maximum(cpm_baseline, cpm_apap) >= 1.0) & ((count_baseline + count_apap) >= 10.0)
    baseline_log2cpm = np.log2(cpm_baseline[keep] + 1)
    apap_log2cpm = np.log2(cpm_apap[keep] + 1)
    return pd.DataFrame({"gene": var_names[keep], "pair": pair_name, "relative_state": state, "baseline_log2CPM": baseline_log2cpm, "APAP_log2CPM": apap_log2cpm, "delta_log2CPM": apap_log2cpm - baseline_log2cpm})

effects = pd.concat([zone_effect_table(pair, state) for pair in PAIRS for state in RELATIVE_STATES], ignore_index=True)
primary = effects.query("pair == 'MN1_to_MN2'")
pv = primary.query("relative_state == 'Relative PV-like'").set_index("gene")
cv = primary.query("relative_state == 'Relative CV-like'").set_index("gene")
common = pv.index.intersection(cv.index)
zone_effects = pd.DataFrame({
    "gene": common,
    "delta_relative_PV": pv.loc[common, "delta_log2CPM"].to_numpy(),
    "delta_relative_CV": cv.loc[common, "delta_log2CPM"].to_numpy(),
    "mean_expression_PV_MN1": pv.loc[common, "baseline_log2CPM"].to_numpy(),
    "mean_expression_PV_MN2": pv.loc[common, "APAP_log2CPM"].to_numpy(),
    "mean_expression_CV_MN1": cv.loc[common, "baseline_log2CPM"].to_numpy(),
    "mean_expression_CV_MN2": cv.loc[common, "APAP_log2CPM"].to_numpy(),
})
zone_effects["delta_CV_minus_PV"] = zone_effects["delta_relative_CV"] - zone_effects["delta_relative_PV"]
x = zone_effects["delta_relative_PV"].to_numpy(); y = zone_effects["delta_relative_CV"].to_numpy()
xpad = max(np.ptp(x) * 0.12, 0.5); ypad = max(np.ptp(y) * 0.12, 0.5)
xlim = (x.min() - xpad, x.max() + xpad); ylim = (y.min() - ypad, y.max() + ypad)
fig, ax = plt.subplots(figsize=(7.8, 7.2))
ax.scatter(x, y, s=7, alpha=0.14, color="#777777", linewidths=0, rasterized=True)
ax.axhline(0, color="#B8B8B8", lw=0.8); ax.axvline(0, color="#B8B8B8", lw=0.8)
lo, hi = max(xlim[0], ylim[0]), min(xlim[1], ylim[1]); ax.plot([lo, hi], [lo, hi], ls="--", color="#929292", lw=0.9)
offsets = {"Gdf15": (-34, 10), "Fgf21": (-44, -18), "Atf3": (-7, 13), "Hmox1": (-47, -8), "Ddit3": (8, 9), "Nqo1": (8, -13), "Cyp2e1": (8, 8), "Glul": (8, -14), "Cyp2f2": (-34, 8), "Sds": (8, 8)}
indexed = zone_effects.set_index("gene")
for gene, offset in offsets.items():
    if gene not in indexed.index: continue
    gx, gy = float(indexed.loc[gene, "delta_relative_PV"]), float(indexed.loc[gene, "delta_relative_CV"])
    color = "#C65F57" if gene in {"Hmox1", "Atf3", "Ddit3", "Fgf21", "Gdf15", "Nqo1"} else "#537FA8"
    ax.scatter(gx, gy, s=34, color=color, edgecolor="white", linewidth=0.5, zorder=3)
    annotation = ax.annotate(gene, (gx, gy), xytext=offset, textcoords="offset points", fontsize=8, color="#222222")
    annotation.set_path_effects([pe.withStroke(linewidth=2.2, foreground="white")])
ax.set_xlim(*xlim); ax.set_ylim(*ylim)
ax.set_xlabel("Relative PV-like: MN2 - MN1 Δ log2(CPM+1)"); ax.set_ylabel("Relative CV-like: MN2 - MN1 Δ log2(CPM+1)")
ax.set_title("Zone-stratified transcriptional effect sizes", pad=12, fontsize=12)
ax.text(0.99, 0.01, "Descriptive; no replicate-level inference", transform=ax.transAxes, ha="right", va="bottom", fontsize=7.5, color="#666666")
ax.legend(handles=[Line2D([0], [0], marker="o", ls="none", color="#C65F57", label="APAP / stress genes", markersize=5), Line2D([0], [0], marker="o", ls="none", color="#537FA8", label="Zonation-associated markers", markersize=5)], frameon=False, loc="upper left", fontsize=8)
fig.subplots_adjust(left=0.15, right=0.96, bottom=0.13, top=0.91)
f5.save_png(OUTPUT_DIR, fig, "Fig5G_ZoneStratified_Transcriptional_Effect_Sizes.png"); plt.close(fig)
print("Genes in effect-size panel:", len(zone_effects))


Genes in effect-size panel: 15246


### 中文
对每个模块基因使用 MN1 spots 的均值和标准差固定标准化 MN1/MN2，再按相对分区汇总 spot-level 模块分数。

### English
For every module gene, use MN1-spot mean and SD to standardize both MN1 and MN2, then summarize spot-level module scores by relative stratum.

In [12]:
# MN1-anchored APAP module scoring across relative zonation strata
module_summary_rows = []
module_scores = {"MN1": {}, "MN2": {}}
for module, requested in APAP_MODULES.items():
    available = [gene for gene in requested if gene in frames["MN1"] and gene in frames["MN2"]]
    mn1_mean = frames["MN1"][available].mean()
    mn1_sd = frames["MN1"][available].std(ddof=0)
    usable = [gene for gene in available if mn1_sd[gene] > 1e-6]
    if not usable:
        raise ValueError(f"No usable MN1-reference genes for {module}")
    for sample in ["MN1", "MN2"]:
        standardized = (frames[sample][usable] - mn1_mean[usable]) / mn1_sd[usable]
        module_scores[sample][module] = standardized.mean(axis=1)
        for state in RELATIVE_STATES:
            values = module_scores[sample][module][relative[sample]["Relative_state"] == state]
            module_summary_rows.append({"module": module, "sample": sample, "Relative_state": state, "median": values.median(), "mean": values.mean(), "Q1": values.quantile(0.25), "Q3": values.quantile(0.75), "n_spots": int(values.notna().sum())})
module_summary = pd.DataFrame(module_summary_rows)

fig, axes = plt.subplots(2, 2, figsize=(10.4, 7.5), sharex=True)
x = np.arange(3, dtype=float)
for ax, module in zip(axes.ravel(), APAP_MODULES):
    data = module_summary.query("module == @module")
    for sample, color, offset in [("MN1", "#AFC7E8", -0.045), ("MN2", "#E6B0AA", 0.045)]:
        values = data.query("sample == @sample").set_index("Relative_state").reindex(RELATIVE_STATES)
        median = values["median"].to_numpy(float)
        ax.errorbar(x + offset, median, yerr=np.vstack([median - values["Q1"].to_numpy(float), values["Q3"].to_numpy(float) - median]), color=color, marker="o", ms=5, lw=1.5, elinewidth=1, capsize=3)
    ax.axhline(0, color="#B5B5B5", lw=0.8, ls="--")
    ax.set_title(module, fontsize=10); ax.set_ylabel("Healthy-reference anchored module score")
    ax.set_xticks(x, RELATIVE_STATES, rotation=18, ha="right")
fig.legend(handles=[Line2D([0], [0], color="#AFC7E8", marker="o", label="MN1 - Healthy, 0 h"), Line2D([0], [0], color="#E6B0AA", marker="o", label="MN2 - APAP, 6 h")], loc="upper center", bbox_to_anchor=(0.5, 0.925), ncol=2, frameon=False)
fig.suptitle("APAP-associated modules across relative zonation strata", fontsize=12, y=0.985)
fig.text(0.5, 0.945, "MN1 Healthy-reference anchored scores; spot-level descriptive medians and IQR", ha="center", fontsize=8, color="#555555")
fig.subplots_adjust(left=0.09, right=0.98, bottom=0.10, top=0.86, wspace=0.25, hspace=0.38)
f5.save_png(OUTPUT_DIR, fig, "Fig5H_APAP_Module_Response_Across_Relative_Zonation_Strata.png"); plt.close(fig)
print(module_summary.round(3).to_string(index=False))


                  module sample   Relative_state  median   mean     Q1    Q3  n_spots
 Oxidative stress / NRF2    MN1 Relative PV-like  -0.041 -0.031 -0.223 0.174      408
 Oxidative stress / NRF2    MN1     Intermediate  -0.009  0.003 -0.206 0.214      409
 Oxidative stress / NRF2    MN1 Relative CV-like   0.005  0.028 -0.176 0.208      409
 Oxidative stress / NRF2    MN2 Relative PV-like   2.464  2.759  2.049 3.263      615
 Oxidative stress / NRF2    MN2     Intermediate   2.485  2.843  2.013 3.388      615
 Oxidative stress / NRF2    MN2 Relative CV-like   2.364  2.624  1.892 3.073      615
     General / ER stress    MN1 Relative PV-like  -0.090 -0.033 -0.281 0.155      408
     General / ER stress    MN1     Intermediate  -0.075 -0.017 -0.297 0.178      409
     General / ER stress    MN1 Relative CV-like   0.022  0.050 -0.225 0.278      409
     General / ER stress    MN2 Relative PV-like   3.923  3.958  3.163 4.779      615
     General / ER stress    MN2     Intermediate   4.0

### 中文
分别比较 MN1→MN2 与 MN5→MN4、CON3→CON2 在 CV-like/PV-like 层的全基因效应，并计算 Spearman 相关。

### English
Compare MN1→MN2 effects with MN5→MN4 and CON3→CON2 effects in CV-like/PV-like strata using Spearman correlation.

In [13]:
# Supportive zone-stratified effect concordance
supportive_rows = []
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.0))
panels = [("Relative CV-like", "MN5_to_MN4"), ("Relative PV-like", "MN5_to_MN4"), ("Relative CV-like", "CON3_to_CON2"), ("Relative PV-like", "CON3_to_CON2")]
for ax, (state, pair) in zip(axes.ravel(), panels):
    main_effect = effects.query("pair == 'MN1_to_MN2' and relative_state == @state").set_index("gene")["delta_log2CPM"]
    support_effect = effects.query("pair == @pair and relative_state == @state").set_index("gene")["delta_log2CPM"]
    common = main_effect.index.intersection(support_effect.index)
    rho = float(spearmanr(main_effect.loc[common], support_effect.loc[common]).statistic)
    supportive_rows.append({"Relative_state": state, "support_pair": pair, "n_genes": len(common), "Spearman_rho": rho})
    ax.scatter(main_effect.loc[common], support_effect.loc[common], s=5, alpha=0.2, color="#627D98")
    ax.axhline(0, color="#aaa", lw=0.7); ax.axvline(0, color="#aaa", lw=0.7)
    ax.set_title(f"{state} | {pair}\nSpearman rho={rho:.2f}")
    ax.set_xlabel("MN1→MN2 Δ log2CPM"); ax.set_ylabel(f"{pair} Δ log2CPM")
fig.suptitle("Supportive zone-stratified effect concordance\nMN5/MN4 and CON3/CON2 are not matched longitudinal pairs")
fig.tight_layout()
f5.save_png(OUTPUT_DIR, fig, "FigS5I_ZoneStratified_Supportive_Concordance.png"); plt.close(fig)
supportive_concordance = pd.DataFrame(supportive_rows)
print(supportive_concordance.to_string(index=False))


  Relative_state support_pair  n_genes  Spearman_rho
Relative CV-like   MN5_to_MN4    14899      0.684042
Relative PV-like   MN5_to_MN4    14709      0.639144
Relative CV-like CON3_to_CON2    15148      0.732110
Relative PV-like CON3_to_CON2    15048      0.673091


### 中文
为 MN1/MN2 构建对称二值 6-NN 权重，执行999次双侧置换 Moran’s I，并在样本内进行 BH-FDR。

### English
Build symmetric binary 6-NN weights for MN1/MN2, run 999 two-sided Moran’s I permutations, and apply within-sample BH-FDR.

In [14]:
# Global Moran's I with symmetric binary 6-NN weights and 999 permutations
moran_cv_genes = ["Glul", "Cyp2e1", "Cyp1a2"]
moran_pv_genes = ["Cyp2f2", "Sds", "Pck1", "Hal"]
moran_genes = ["Hmox1", "Ddit3", "Fgf21", "Cxcl2", "Atf3", "Glul", "Cyp2e1", "Cyp2f2", "Sds"]
moran_module_scores = {"MN1": {}, "MN2": {}}
moran_zonation_scores = {}
for sample in ["MN1", "MN2"]:
    expr = frames[sample]
    zzon = (expr[moran_cv_genes + moran_pv_genes] - expr[moran_cv_genes + moran_pv_genes].mean()) / expr[moran_cv_genes + moran_pv_genes].std(ddof=0).replace(0, np.nan)
    moran_zonation_scores[sample] = zzon[moran_cv_genes].mean(axis=1) - zzon[moran_pv_genes].mean(axis=1)
    for module, requested in APAP_MODULES.items():
        available = [gene for gene in requested if gene in expr]
        z = (expr[available] - expr[available].mean()) / expr[available].std(ddof=0).replace(0, np.nan)
        moran_module_scores[sample][module] = z.fillna(0).mean(axis=1)

feature_order = moran_genes + ["Zonation_score"] + list(APAP_MODULES)
moran_rows = []
for sample in ["MN1", "MN2"]:
    feature_matrix = np.column_stack([frames[sample][gene] for gene in moran_genes] + [moran_zonation_scores[sample]] + [moran_module_scores[sample][module] for module in APAP_MODULES])
    weights = f5.symmetric_knn(coords[sample], k=6)  # binary; W = max(W, W.T)
    observed, permutation_p = f5.moran_many(feature_matrix, weights, permutations=999, seed=RANDOM_SEED)
    feature_types = ["gene"] * len(moran_genes) + ["score"] + ["module"] * len(APAP_MODULES)
    for feature, feature_type, value, p_value in zip(feature_order, feature_types, observed, permutation_p):
        moran_rows.append({"sample": sample, "feature": feature, "feature_type": feature_type, "Moran_I": value, "p_value": p_value})
moran_results = pd.DataFrame(moran_rows)
moran_results["FDR"] = np.nan
for sample in ["MN1", "MN2"]:
    idx = moran_results.index[moran_results["sample"] == sample]
    moran_results.loc[idx, "FDR"] = multipletests(moran_results.loc[idx, "p_value"], method="fdr_bh")[1]
moran_results["significant_FDR_0.05"] = moran_results["FDR"] < 0.05

groups = {"APAP / stress-associated genes": ["Hmox1", "Ddit3", "Fgf21", "Cxcl2", "Atf3"], "Liver zonation-associated features": ["Glul", "Cyp2e1", "Cyp2f2", "Sds", "Zonation_score"], "Stress / injury modules": list(APAP_MODULES)}
order = [feature for values in groups.values() for feature in values]
positions, separators, cursor = {}, [], 0.0
for group_index, values in enumerate(groups.values()):
    for feature in values:
        positions[feature] = cursor; cursor += 1
    if group_index < len(groups) - 1:
        separators.append(cursor - 0.1); cursor += 0.8
observed_min, observed_max = moran_results["Moran_I"].min(), moran_results["Moran_I"].max()
span = observed_max - observed_min
fig, ax = plt.subplots(figsize=(8.8, 7.1))
for separator in separators: ax.axhline(separator, color="#E3E3E3", linewidth=0.75)
for sample, offset in [("MN1", -0.15), ("MN2", 0.15)]:
    subset = moran_results.query("sample == @sample").set_index("feature").loc[order]
    y = np.array([positions[f] for f in order]) + offset
    for x_value, y_value, significant in zip(subset["Moran_I"], y, subset["significant_FDR_0.05"]):
        ax.scatter(x_value, y_value, s=54, facecolor=SAMPLE_COLORS[sample] if significant else "white", edgecolor=SAMPLE_COLORS[sample], linewidth=1.35, zorder=3)
ax.axvline(0, color="#B7B7B7", linewidth=0.9, linestyle=(0, (4, 3)))
ax.set_xlim(min(-0.025, observed_min - 0.08 * span), observed_max + 0.10 * span)
ax.set_ylim(max(positions.values()) + 0.65, -0.65)
ax.set_yticks([positions[f] for f in order], order)
ax.set_xlabel("Global Moran's I"); ax.xaxis.set_major_locator(MaxNLocator(nbins=6)); ax.tick_params(axis="y", labelsize=9, length=0, pad=6)
fig.legend(handles=[Line2D([0], [0], marker="o", ls="none", markerfacecolor=SAMPLE_COLORS[s], markeredgecolor=SAMPLE_COLORS[s], label=("MN1 — Healthy, 0 h" if s == "MN1" else "MN2 — APAP, 6 h"), markersize=6.3) for s in ["MN1", "MN2"]], title="Sample", frameon=False, loc="upper left", bbox_to_anchor=(0.37, 0.985), ncol=2, fontsize=8.5)
fig.legend(handles=[Line2D([0], [0], marker="o", ls="none", markerfacecolor="#666", markeredgecolor="#666", label="Filled: FDR < 0.05", markersize=6.3), Line2D([0], [0], marker="o", ls="none", markerfacecolor="white", markeredgecolor="#666", label="Open: FDR ≥ 0.05", markersize=6.3)], title="Significance", frameon=False, loc="upper left", bbox_to_anchor=(0.37, 0.91), ncol=2, fontsize=8.5)
fig.subplots_adjust(left=0.37, right=0.985, bottom=0.105, top=0.81)
f5.save_png(OUTPUT_DIR, fig, "FigS5J_Spatial_Autocorrelation_MoransI.png"); plt.close(fig)
print(moran_results.to_string(index=False))


sample                  feature feature_type   Moran_I  p_value      FDR  significant_FDR_0.05
   MN1                    Hmox1         gene -0.005407    0.726 0.781846                 False
   MN1                    Ddit3         gene  0.009838    0.519 0.660545                 False
   MN1                    Fgf21         gene  0.017466    0.282 0.394800                 False
   MN1                    Cxcl2         gene -0.007223    0.875 0.875000                 False
   MN1                     Atf3         gene  0.033264    0.025 0.050000                 False
   MN1                     Glul         gene  0.208648    0.001 0.002800                  True
   MN1                   Cyp2e1         gene  0.214851    0.001 0.002800                  True
   MN1                   Cyp2f2         gene  0.152866    0.001 0.002800                  True
   MN1                      Sds         gene  0.102004    0.001 0.002800                  True
   MN1           Zonation_score        score  0.16

### 中文
列出本次运行生成的18张 PNG，并确认输出目录没有其他 PNG。

### English
List the 18 PNGs generated in this run and confirm that the output directory contains no additional PNGs.

In [15]:
# Output summary
output_summary = f5.output_inventory(OUTPUT_DIR)
print(output_summary.to_string(index=False))


                                                        filename   bytes
                                    Fig5B_MN2_vs_MN1_Volcano.png 2138537
                                                Fig5C1_GO_Up.png  341257
                                              Fig5C2_GO_Down.png  304048
                   Fig5D_APAP_Injury_Gene_Spatial_Maps_fixed.png 5269513
                    Fig5E_Liver_Zonation_Marker_Spatial_Maps.png 5303433
           Fig5F_Relative_Zonation_3State_Spatial_Maps_fixed.png 3404566
           Fig5G_ZoneStratified_Transcriptional_Effect_Sizes.png  629239
  Fig5H_APAP_Module_Response_Across_Relative_Zonation_Strata.png  565781
                                             FigS5A1_KEGG_Up.png  288899
                                           FigS5A2_KEGG_Down.png  285508
            FigS5B_APAP_Marker_Expression_Distribution_fixed.png  223766
              FigS5C_Zonation_Marker_Expression_Distribution.png  212838
      FigS5D_Relative_Zonation_Continuous_Spatial_M